# 01. Data curation, PHI screening and corpus governance

**Goal.** Turn raw documents into a governed, versioned corpus that is safe to index.

**Playbook steps covered:** 1 (use case and risk class), 2 (acquire and curate corpus)

> Every step below has three parts: a **rationale** (why we do it), the **code** with line-level comments
> (how we do it), and an **evaluation** (how we know it worked). All notebooks run **offline** by default
> (hashing embedder + extractive baseline LLM). Switch to Claude and sentence-transformers with
> `AD_RAG_LLM_BACKEND=anthropic` and `AD_RAG_EMBEDDING_BACKEND=sentence-transformers`.

> The corpus is **synthetic educational content**. Nothing here is clinical guidance.

In [1]:
# ---- Setup: make the notebook runnable from anywhere (GitHub clone, Colab, Docker) ----
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Find the repo root by walking up until we see configs/config.yaml.
ROOT = Path.cwd()
while not (ROOT / "configs" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.environ["AD_RAG_HOME"] = str(ROOT)          # tell the package where the repo lives

# Fresh environment (e.g. Colab)? Install the package in editable mode first:
#   %pip install -q -e ".[notebooks]"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ad_rag import load_settings, build_pipeline

settings = load_settings()                     # typed config from configs/config.yaml (+ env overrides)
print("repo root :", ROOT)
print("embedder  :", settings.embedding.backend, "| llm backend:", settings.llm.backend)

repo root : /home/yodeyemi/alzheimers-llm-rag-agent
embedder  : hashing | llm backend: extractive


### Step 1: Define the intended use and risk class

**Rationale.** A precise intended-use statement decides what data we need, which guardrails we need and whether the system drifts toward regulated device territory. Writing it as DATA (not prose) lets tests and governance reviews reference one source of truth.

**How this step is evaluated.** Reviewed and signed by clinical, regulatory and quality owners. Any scope change re-opens this review.

In [2]:
# The intended-use statement as a machine-readable object. Governance docs and tests import the same facts.
INTENDED_USE = {
    "product": "AD Clinical Evidence Assistant",
    "users": ["qualified clinicians", "medical affairs staff"],
    "in_scope": ["explain published guidance on blood biomarker use",
                 "summarise internal assay validation dossiers with citations",
                 "interpret an assay value against configured cutoffs (deterministic tool)"],
    "out_of_scope": ["individual diagnosis", "individual dosing or treatment advice",
                     "interpretation of a named patient's results", "asymptomatic screening advice"],
    "risk_posture": "Non-device clinical decision support with a human in the loop. "
                    "A regulated SaMD assessment is required BEFORE any scope expansion.",
}
pd.DataFrame({"field": list(INTENDED_USE), "value": [str(v) for v in INTENDED_USE.values()]})

,field,value
0,product,AD Clinical Evidence Assistant
1,users,"['qualified clinicians', 'medical affairs staff']"
2,in_scope,['explain published guidance on blood biomarke...
3,out_of_scope,"['individual diagnosis', 'individual dosing or..."
4,risk_posture,Non-device clinical decision support with a hu...


In [3]:
# Evaluation of this step: the out-of-scope list must cover the prohibited uses in our policy document.
policy = (settings.path("corpus_dir") / "ai_assistant_use_policy.md").read_text().lower()
for item in ["diagnose individuals", "individual dosing"]:
    assert item in policy, f"policy does not mention: {item}"
assert "individual diagnosis" in INTENDED_USE["out_of_scope"]
print("Intended use is consistent with the written AI use policy.")

Intended use is consistent with the written AI use policy.


### Step 2: Load the corpus and build a source inventory

**Rationale.** RAG quality is capped by corpus quality. Before anything else we need to know exactly what is in the corpus: title, type, version, licence and size. This inventory is a governance artifact (provenance and licensing).

**How this step is evaluated.** Every document must have a licence and a version. Unknown licences block indexing.

In [4]:
from ad_rag.ingestion import load_documents, inventory, corpus_version

docs = load_documents(settings.path("corpus_dir"))      # reads .md/.txt/.pdf, parses the front-matter block
inv = pd.DataFrame(inventory(docs))                      # provenance table: id, title, type, version, licence, words
display(inv)
print("documents:", len(docs), "| total words:", int(inv.words.astype(int).sum()))

# Evaluation: no document may have an unknown licence or version.
assert (inv.license != "unknown").all() and (inv.version != "n/a").all()
print("Licence and version present for every document.")

,doc_id,title,source_type,version,license,words
0,ai_assistant_use_policy,AI Assistant Use Policy,synthetic_policy,1.0,CC0-1.0 (synthetic),164
1,anti_amyloid_therapy_considerations,Considerations for Anti-Amyloid Therapy,synthetic_guideline,1.0,CC0-1.0 (synthetic),155
2,appropriate_use_criteria,Appropriate Use Criteria for Blood Biomarker T...,synthetic_policy,1.0,CC0-1.0 (synthetic),185
3,assay_validation_summary,Validation Summary for a Fictional Assay (Demo...,synthetic_dossier,1.0,CC0-1.0 (synthetic),144
4,blood_biomarker_overview,Blood Biomarkers in Alzheimer's Disease: Overview,synthetic_overview,1.0,CC0-1.0 (synthetic),188
5,confounders_and_comorbidities,Confounders and Comorbidities Affecting Plasma...,synthetic_guideline,1.0,CC0-1.0 (synthetic),152
6,data_privacy_and_phi_handling,Data Privacy and PHI Handling,synthetic_policy,1.0,CC0-1.0 (synthetic),122
7,diagnostic_pathway_primary_care,Diagnostic Pathway in Primary Care,synthetic_guideline,1.0,CC0-1.0 (synthetic),166
8,disease_staging_and_definitions,Disease Staging and Definitions,synthetic_overview,1.0,CC0-1.0 (synthetic),163
9,two_cutoff_interpretation,Two-Cutoff Interpretation of Biomarker Results,synthetic_guideline,1.0,CC0-1.0 (synthetic),176


documents: 10 | total words: 1615
Licence and version present for every document.


### Step 3: Screen for PHI before anything is embedded

**Rationale.** Protected health information must never enter the vector index, prompts, logs or traces. Embeddings can leak the text they encode, so the gate must sit BEFORE embedding. Regex rules catch structured identifiers (SSN, phone, email, MRN, DOB, labelled names). In production add NER-based screening (Microsoft Presidio) behind the same interface.

**How this step is evaluated.** 100% of seeded PHI examples are detected (recall on a synthetic test set) and zero clean documents are quarantined (no false positives on the corpus).

In [5]:
from ad_rag.phi import scan_text, redact

# Synthetic PHI examples (fake identifiers) plus one clean sentence as a negative control.
samples = [
    "Patient: Jane Doe, MRN 1234567, DOB: 04/05/1950 presented with memory loss.",
    "Call 555-123-4567 or email jane.doe@example.com about the follow-up.",
    "SSN 123-45-6789 was entered in the free-text field.",
    "Plasma p-tau217 is a marker of amyloid-related pathology.",          # clean negative control
]
rows = []
for text in samples:
    findings = scan_text(text)                           # list of (kind, span) findings
    rows.append({"text": text, "found": [f.kind for f in findings], "redacted": redact(text)})
pd.set_option("display.max_colwidth", 90)
display(pd.DataFrame(rows))
assert rows[-1]["found"] == []                           # negative control must stay clean
assert all(r["found"] for r in rows[:-1])                # every seeded example must be caught

,text,found,redacted
0,"Patient: Jane Doe, MRN 1234567, DOB: 04/05/1950 presented with memory loss.","[PATIENT_NAME, MRN, DOB]","[REDACTED:PATIENT_NAME], [REDACTED:MRN], [REDACTED:DOB] presented with memory loss."
1,Call 555-123-4567 or email jane.doe@example.com about the follow-up.,"[PHONE, EMAIL]",Call [REDACTED:PHONE] or email [REDACTED:EMAIL] about the follow-up.
2,SSN 123-45-6789 was entered in the free-text field.,[SSN],SSN [REDACTED:SSN] was entered in the free-text field.
3,Plasma p-tau217 is a marker of amyloid-related pathology.,[],Plasma p-tau217 is a marker of amyloid-related pathology.


In [6]:
from ad_rag.indexing import phi_gate
from ad_rag.ingestion import Document

# The gate separates clean documents from quarantined ones. We add one "poisoned" document to prove it works.
poisoned = Document("poisoned_note", "Intake note", "poisoned_note.md",
                    "Intake note. Patient: John Smith MRN 7654321 reported memory decline.")
clean, quarantined = phi_gate(docs + [poisoned])

print("clean documents      :", len(clean))
print("quarantined documents:", quarantined)

# Evaluation: only the poisoned document is blocked; the real corpus passes untouched.
assert [q["doc_id"] for q in quarantined] == ["poisoned_note"]
assert len(clean) == len(docs)
print("PHI gate OK: blocked the poisoned note, passed all real documents.")

clean documents      : 10
quarantined documents: [{'doc_id': 'poisoned_note', 'kinds': ['MRN', 'PATIENT_NAME'], 'n_findings': 2}]
PHI gate OK: blocked the poisoned note, passed all real documents.


### Step 4: Audit coverage against the golden question set

**Rationale.** A RAG system can only answer what the corpus covers. Mapping every golden question to its expected source document exposes coverage gaps BEFORE users find them, and reveals documents nobody asks about (candidates for pruning or for new test questions).

**How this step is evaluated.** Every expected source exists in the corpus (no dangling references). Report documents with zero questions.

In [7]:
from ad_rag.evaluation import load_golden

golden = load_golden(ROOT / "data" / "eval" / "golden_set.jsonl")
in_scope = [g for g in golden if g["category"] == "in_scope"]

doc_sources = {d.source for d in docs}
expected = {s for g in in_scope for s in g["expected_sources"]}
missing = expected - doc_sources
assert not missing, f"golden set references documents that do not exist: {missing}"

# Coverage table: how many questions and topics point at each document.
coverage = (pd.DataFrame(in_scope).explode("expected_sources")
            .groupby("expected_sources")
            .agg(n_questions=("id", "count"), topics=("topic", lambda s: ", ".join(sorted(set(s))))))
display(coverage)
print("documents with no golden question:", sorted(doc_sources - set(coverage.index)) or "none")

,n_questions,topics
expected_sources,,
ai_assistant_use_policy.md,1,governance
anti_amyloid_therapy_considerations.md,3,therapy
appropriate_use_criteria.md,4,appropriate_use
assay_validation_summary.md,2,assay
blood_biomarker_overview.md,2,overview
confounders_and_comorbidities.md,3,"confounders, fairness"
data_privacy_and_phi_handling.md,2,governance
diagnostic_pathway_primary_care.md,3,pathway
disease_staging_and_definitions.md,2,definitions


documents with no golden question: none


### Step 5: Version the corpus

**Rationale.** Every answer must be traceable to the exact corpus snapshot that produced it (audit, incident review, reproducibility). We hash document IDs and contents into a short `corpus_version` that is stored in the index manifest, returned with every answer and written to the audit log.

**How this step is evaluated.** The hash is deterministic (same corpus gives the same version) and changes when ANY document changes.

In [8]:
v1 = corpus_version(docs)
v1_again = corpus_version(list(reversed(docs)))          # order must not matter

# Simulate an edit to the first document and recompute.
edited = [Document(d.doc_id, d.title, d.source, d.text + ("\nEdited." if i == 0 else ""), d.meta)
          for i, d in enumerate(docs)]
v2 = corpus_version(edited)

print("version        :", v1)
print("same corpus    :", v1_again, "(identical)")
print("after one edit :", v2)
assert v1 == v1_again and v1 != v2

version        : e639851bb748
same corpus    : e639851bb748 (identical)
after one edit : b80fd0d016f7


## Summary and exit criteria for this notebook

| Check | Result |
|---|---|
| Intended-use statement consistent with policy | asserted in Step 1 |
| Licence and version present for every document | asserted in Step 2 |
| PHI gate catches seeded identifiers, no false positives on corpus | asserted in Step 3 |
| Golden questions map to existing documents | asserted in Step 4 |
| Corpus version deterministic and change-sensitive | asserted in Step 5 |

**Next:** `02_indexing_and_retrieval.ipynb` chunks, embeds and indexes this corpus.